# Coronary Artery Stenosis Detection using YOLOv8 (v2 — Accuracy Phase)

End-to-end pipeline to detect **coronary artery stenosis** in X-ray angiography images from the **ARCADE** dataset
(COCO-format annotations, single class: `stenosis`, `category_id = 26`) using **YOLOv8 (Ultralytics, PyTorch)**.

**This is a cleaned rewrite of the original experiment notebook.** Scope for this pass: maximize detection accuracy
on the official ARCADE stenosis split with a correct, defensible pipeline — no explainability module yet (that is
Objective 2 in the proposal) and no cross-dataset transfer yet (Objective 3). Those are added on top of this once the
accuracy baseline here is solid, so results are comparable stage-by-stage in the thesis.

**What changed vs. the original notebook (and why — bugs, not opinions):**
1. The 3-channel preprocessing described in Section 4's markdown was **never actually applied** — `DEFAULT_CONFIG`
   had every stage set to `False`, so the "enhanced" images were identical to the raw images. This is the single
   biggest correctness bug in the original notebook: the whole preprocessing section ran and produced plots, but had
   zero effect on what YOLO trained on.
2. Preprocessing is now genuinely **multi-cue**: channel 0 = denoised grayscale, channel 1 = CLAHE local-contrast
   enhancement, channel 2 = a vesselness filter (Sato, tuned for thin tubular structures). Three different views of
   the same artery instead of one filter chain duplicated three times.
3. Because channels 1 and 2 are no longer literal color information, **HSV color-jitter augmentation is disabled**
   (`hsv_s=0`, `hsv_v=0`). Randomly jittering "saturation/value" of a vesselness map does not correspond to anything
   physical and just adds noise to the label.
4. Image size raised from 640 → 960. Stenosis lesions are small relative to the full angiogram; Section 3 below
   quantifies this from the actual annotation boxes rather than assuming it.
5. Dead/duplicate cells removed (an old commented-out preprocessing block, a stray `ray` install/uninstall detour,
   an incomplete `top_hat` function that was cut off mid-line in the source file).
6. Dataset acquisition rewritten for **Google Colab** (the original hardcoded a Kaggle `/kaggle/input/...` path).
7. An **ablation switch** (`ENABLE_PREPROCESSING`) is added so you can run the pipeline with preprocessing on and off
   and report the accuracy delta — this is genuine thesis evidence that the preprocessing step earns its place in
   Objective 1, not just a claim.

**Pipeline overview**
1. Install & import dependencies
2. Get the ARCADE dataset onto the Colab runtime
3. Explore the COCO-format dataset + quantify lesion size relative to image size
4. Preprocess images (CLAHE + vesselness stack) to enhance vessel/lesion visibility
5. Convert COCO annotations → YOLO `.txt` label format
6. Build the YOLO dataset folder structure + `data.yaml`
7. Configure augmentation (tuned for angiography, not generic photos)
8. Train YOLOv8
9. Evaluate with standard detection metrics (Precision/Sensitivity, PPV, F1, mAP@0.5, mAP@0.5:0.95, confusion matrix)
10. Manual IoU-matching cross-check (independent of Ultralytics' internal NMS/metric code)
11. Visualize predictions vs. ground truth
12. Save the trained weights to Google Drive so they survive the Colab session ending

> Designed to run on **Google Colab** with a GPU runtime (Runtime → Change runtime type → T4 GPU, or better).


## 1. Install & Import Dependencies

In [ ]:
# ============================================================
# 1a. Installs
# ============================================================
!pip install -q ultralytics pycocotools scikit-image kagglehub
print("Installs complete.")


In [ ]:
# ============================================================
# 1b. Imports
# ============================================================
import os
import json
import shutil
import random
import glob
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from skimage.filters import frangi, sato, meijering, hessian
from skimage import exposure

from ultralytics import YOLO, settings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Ultralytics ships a Ray Tune callback that fires every epoch even when Ray Tune
# isn't being used. On Colab the bundled `ray` version sometimes doesn't match what
# that callback expects and training crashes with an AttributeError. Disabling it
# has no effect on training results/metrics -- it just isn't invoked.
settings.update({"raytune": False})


## 2. Get the ARCADE Dataset onto the Colab Runtime

The dataset lives on Kaggle: https://www.kaggle.com/datasets/nirmalgaud/arcade-dataset

`kagglehub` downloads it straight into the Colab runtime's local disk without needing you to manually upload a zip.
The first time you run this in a fresh Colab session it will ask you to authenticate with a Kaggle account (either a
`kaggle.json` API token you upload, or a browser login) — you only need to do this once per session.

We then **search** the downloaded tree for `train.json` rather than hardcoding a path, because the exact folder
depth `kagglehub` produces can vary by dataset version. This makes the notebook resilient to that instead of silently
pointing at the wrong directory.


In [ ]:
# ============================================================
# 2a. Download dataset via kagglehub
# ============================================================
import kagglehub

download_path = Path(kagglehub.dataset_download("nirmalgaud/arcade-dataset"))
print("Downloaded to:", download_path)

# Locate the folder that directly contains train/val/test with annotations/train.json etc.
candidates = list(download_path.rglob("train.json"))
if not candidates:
    raise FileNotFoundError(
        f"Could not find train.json anywhere under {download_path}. "
        "Browse the folder manually with `!find {download_path}` and set DATASET_ROOT by hand."
    )

# train.json is expected at <DATASET_ROOT>/train/annotations/train.json
DATASET_ROOT = candidates[0].parent.parent.parent
print("Resolved DATASET_ROOT:", DATASET_ROOT)

for split in ["train", "val", "test"]:
    ann = DATASET_ROOT / split / "annotations" / f"{split}.json"
    img_dir = DATASET_ROOT / split / "images"
    assert ann.exists(), f"Missing {ann}"
    assert img_dir.exists(), f"Missing {img_dir}"
print("Dataset structure verified for train/val/test.")


In [ ]:
# ============================================================
# 2b. (Optional) Mount Google Drive so trained weights / results survive
#     after the Colab runtime is recycled. Skip this cell if you'd rather
#     download files manually from the Colab file browser at the end.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/arcade_stenosis_runs")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive output dir:", DRIVE_OUTPUT_DIR)


## 3. Configuration

In [ ]:
# ============================================================
# 3. Configuration
# ============================================================
WORK_ROOT = Path("/content/stenosis_yolo")
PREPROC_ROOT = WORK_ROOT / "preprocessed_images"
YOLO_ROOT = WORK_ROOT / "yolo_dataset"

SPLITS = ["train", "val", "test"]

STENOSIS_CATEGORY_ID = 26
CLASS_NAMES = ["stenosis"]

# --- Ablation switch: run once with True, once with False, compare the two
#     final_test_metrics.csv files. This is the accuracy evidence for the
#     preprocessing step, not just an assumption. ---
ENABLE_PREPROCESSING = True
RUN_TAG = "enhanced" if ENABLE_PREPROCESSING else "raw"

# imgsz raised from the original 640 -- Section 4 below measures why.
IMG_SIZE = 960
BATCH_SIZE = 16          # drop to 8 if you hit CUDA out-of-memory at imgsz=960 on a T4
EPOCHS = 150
MODEL_WEIGHTS = "yolov8s.pt"   # yolov8s: best accuracy/overfit trade-off for ~1500 images.
                                 # Try yolov8m.pt as a second run only if val mAP is still
                                 # climbing at EPOCHS with yolov8s (more capacity, more overfit risk).

if torch.cuda.is_available():
    DEVICE = 0
else:
    DEVICE = "cpu"
    print("WARNING: no GPU detected -- training at imgsz=960 for 150 epochs on CPU will be very slow. "
          "Go to Runtime > Change runtime type > T4 GPU.")

for p in [WORK_ROOT, PREPROC_ROOT, YOLO_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("Work root:", WORK_ROOT)
print("Using device:", DEVICE)
print("Preprocessing enabled:", ENABLE_PREPROCESSING, "| run tag:", RUN_TAG)


## 4. Explore the COCO-format Dataset

Load each split's JSON, confirm the schema, count images/annotations, and — importantly for choosing `IMG_SIZE`
above — measure how large stenosis boxes actually are relative to the full image. If lesions occupy a tiny fraction
of the frame, that's the quantitative justification for training at a larger resolution instead of the YOLO default
of 640, and it's the kind of number a thesis committee will ask for if you claim "small lesions are hard to detect."


In [ ]:
# ============================================================
# 4a. Load & inspect COCO annotations for each split
# ============================================================
def load_coco(split):
    ann_path = DATASET_ROOT / split / "annotations" / f"{split}.json"
    with open(ann_path, "r") as f:
        return json.load(f)

coco_data = {split: load_coco(split) for split in SPLITS}

summary_rows = []
for split, coco in coco_data.items():
    n_images = len(coco["images"])
    n_anns = len(coco["annotations"])
    cat_ids = sorted(set(a["category_id"] for a in coco["annotations"]))
    n_imgs_with_ann = len(set(a["image_id"] for a in coco["annotations"]))
    summary_rows.append({
        "split": split,
        "num_images": n_images,
        "num_annotations": n_anns,
        "num_images_with_annotations": n_imgs_with_ann,
        "pct_images_with_stenosis": round(100 * n_imgs_with_ann / n_images, 1),
        "category_ids_present": cat_ids,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

print("\nCategories (from train.json):")
for c in coco_data["train"]["categories"]:
    print(c)


In [ ]:
# ============================================================
# 4b. Lesion size relative to image size -- justifies IMG_SIZE choice
# ============================================================
size_rows = []
for split, coco in coco_data.items():
    img_wh = {im["id"]: (im["width"], im["height"]) for im in coco["images"]}
    for ann in coco["annotations"]:
        if ann["category_id"] != STENOSIS_CATEGORY_ID:
            continue
        img_w, img_h = img_wh[ann["image_id"]]
        _, _, w, h = ann["bbox"]
        box_area_frac = (w * h) / (img_w * img_h)
        size_rows.append({"split": split, "box_w_px": w, "box_h_px": h,
                           "img_w": img_w, "img_h": img_h,
                           "box_area_pct_of_image": 100 * box_area_frac})

size_df = pd.DataFrame(size_rows)
print(size_df.groupby("split")["box_area_pct_of_image"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

median_pct = size_df["box_area_pct_of_image"].median()
print(f"\nMedian stenosis box covers only {median_pct:.2f}% of the image area.")
print("This is the quantitative basis for training at IMG_SIZE=960 rather than YOLO's default 640: "
      "at 640px, a box this small can shrink to only a handful of pixels after the backbone's downsampling, "
      "which is exactly where small-object detectors lose recall.")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(size_df["box_area_pct_of_image"], bins=40)
ax.set_xlabel("Stenosis box area as % of full image area")
ax.set_ylabel("Count")
ax.set_title("Distribution of stenosis lesion size relative to image")
plt.tight_layout()
plt.show()


## 5. Preprocessing: Making the Lesions Clearer

Coronary X-ray angiograms are low-contrast, noisy, and the vessels are thin tubular structures. Instead of one
filter chain duplicated across 3 identical channels (what the original notebook's code effectively did once you
account for the disabled config), each output channel now carries genuinely different information:

- **Channel 0 — denoised grayscale.** Mild non-local-means denoising, preserves the raw intensity signal so nothing
  is thrown away.
- **Channel 1 — CLAHE.** Locally boosts contrast so faint lesions in dim regions of the frame become visible without
  blowing out already-bright regions.
- **Channel 2 — vesselness map (Sato filter).** A Hessian-based multiscale filter that responds specifically to
  tubular structures and suppresses blob-like noise — this is the standard technique in vascular image analysis for
  making the artery tree itself pop out from the background. Sato is used here rather than Frangi because Sato is
  less sensitive to the noise floor typical of clinical angiograms at the vessel widths present in ARCADE; both are
  implemented below so you can swap `VESSEL_METHOD` and compare.

Feeding YOLO's 3-channel input three different *views* of the same anatomy — rather than the same filtered image
copied 3 times — gives the network more to learn from at essentially no extra compute cost.


In [ ]:
# ============================================================
# 5a. Preprocessing building blocks
# ============================================================
def to_grayscale(img):
    if img.ndim == 3:
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img


def normalize(gray):
    return cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


def denoise(gray, method="nlm"):
    if method == "nlm":
        return cv2.fastNlMeansDenoising(gray, None, 7, 7, 21)
    elif method == "bilateral":
        return cv2.bilateralFilter(gray, 5, 50, 50)
    elif method == "median":
        return cv2.medianBlur(gray, 3)
    elif method == "gaussian":
        return cv2.GaussianBlur(gray, (3, 3), 0)
    return gray


def apply_clahe(gray, clip_limit=2.5, tile_grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(gray)


def vessel_filter(gray, method="sato", sigma_min=1, sigma_max=5):
    img_f = gray.astype(np.float32) / 255.0
    sigmas = range(sigma_min, sigma_max)
    if method == "frangi":
        vmap = frangi(img_f, sigmas=sigmas, alpha=0.5, beta=0.5, black_ridges=False)
    elif method == "sato":
        vmap = sato(img_f, sigmas=sigmas, black_ridges=False)
    elif method == "meijering":
        vmap = meijering(img_f, sigmas=sigmas, black_ridges=False)
    elif method == "hessian":
        vmap = hessian(img_f, sigmas=sigmas, black_ridges=False)
    else:
        raise ValueError(f"Unknown vessel filter method: {method}")
    vmap = exposure.rescale_intensity(vmap, out_range=(0, 1))
    return (vmap * 255).astype(np.uint8)


VESSEL_METHOD = "sato"
CLAHE_CLIP = 2.5
CLAHE_GRID = (8, 8)
DENOISE_METHOD = "nlm"


def build_enhanced_image(img_bgr):
    """Returns a 3-channel HxWx3 uint8 image:
         ch0 = denoised grayscale, ch1 = CLAHE, ch2 = vesselness map.
       All three are derived from the same denoised base so they stay spatially aligned."""
    gray = to_grayscale(img_bgr)
    gray = normalize(gray)
    den = denoise(gray, DENOISE_METHOD)
    clahe_ch = apply_clahe(den, CLAHE_CLIP, CLAHE_GRID)
    vessel_ch = vessel_filter(den, VESSEL_METHOD)
    return np.stack([den, clahe_ch, vessel_ch], axis=-1)


def preprocess_image(img_bgr):
    """Single entry point used everywhere below. Respects ENABLE_PREPROCESSING
       so the ablation switch in Section 3 actually does something."""
    if not ENABLE_PREPROCESSING:
        return img_bgr
    return build_enhanced_image(img_bgr)


In [ ]:
# ============================================================
# 5b. Visualize preprocessing on a few sample images
# ============================================================
def imread_any(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return img

sample_split = "train"
sample_img_dir = DATASET_ROOT / sample_split / "images"
sample_files = sorted(os.listdir(sample_img_dir))[:3]

fig, axes = plt.subplots(len(sample_files), 5, figsize=(20, 4 * len(sample_files)))
if len(sample_files) == 1:
    axes = axes[None, :]

for i, fname in enumerate(sample_files):
    img = imread_any(sample_img_dir / fname)
    gray = to_grayscale(img)
    gray = normalize(gray)
    den = denoise(gray, DENOISE_METHOD)
    clahe_img = apply_clahe(den, CLAHE_CLIP, CLAHE_GRID)
    vmap = vessel_filter(den, VESSEL_METHOD)
    enhanced = np.stack([den, clahe_img, vmap], axis=-1)

    axes[i, 0].imshow(gray, cmap="gray"); axes[i, 0].set_title(f"{fname}\nOriginal (gray)")
    axes[i, 1].imshow(den, cmap="gray"); axes[i, 1].set_title("Denoised (ch0)")
    axes[i, 2].imshow(clahe_img, cmap="gray"); axes[i, 2].set_title("CLAHE (ch1)")
    axes[i, 3].imshow(vmap, cmap="gray"); axes[i, 3].set_title(f"{VESSEL_METHOD} vesselness (ch2)")
    axes[i, 4].imshow(enhanced); axes[i, 4].set_title("Stacked 3-channel input")
    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()


## 6. Apply Preprocessing to the Whole Dataset

In [ ]:
# ============================================================
# 6. Apply preprocessing to all images, save to PREPROC_ROOT/<split>/
# ============================================================
def preprocess_split(split):
    src_dir = DATASET_ROOT / split / "images"
    dst_dir = PREPROC_ROOT / split
    dst_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(os.listdir(src_dir))
    for fname in tqdm(files, desc=f"Preprocessing {split}"):
        src_path = src_dir / fname
        img = imread_any(src_path)
        enhanced = preprocess_image(img)
        out_path = dst_dir / fname
        if enhanced.ndim == 3 and ENABLE_PREPROCESSING:
            cv2.imwrite(str(out_path), enhanced)   # already channel-stacked, no BGR/RGB swap needed
        else:
            cv2.imwrite(str(out_path), enhanced)   # raw path: still BGR, cv2.imwrite expects that

for split in SPLITS:
    preprocess_split(split)

print("Preprocessing complete. Enhanced images saved under:", PREPROC_ROOT)


## 7. Convert COCO Annotations to YOLO Format

YOLO (Ultralytics) expects, per image, a `.txt` file with one row per object:

```
class_id x_center y_center width height
```

all normalized to `[0, 1]` relative to image width/height. We use the `bbox` field (`[x_min, y_min, width, height]`,
absolute pixels, COCO convention) from each annotation with `category_id == 26` and remap it to class index `0`
(`stenosis`). Images with no stenosis annotation get an empty label file, which YOLO treats as a negative/background
image — that's intentional, not a bug, since it teaches the model what "no lesion" looks like.


In [ ]:
# ============================================================
# 7. COCO bbox -> YOLO bbox conversion
# ============================================================
def coco_bbox_to_yolo(bbox, img_w, img_h):
    x_min, y_min, w, h = bbox
    x_center = x_min + w / 2.0
    y_center = y_min + h / 2.0
    return (x_center / img_w, y_center / img_h, w / img_w, h / img_h)

def convert_split_to_yolo_labels(split, coco, label_out_dir):
    label_out_dir.mkdir(parents=True, exist_ok=True)

    images_by_id = {im["id"]: im for im in coco["images"]}
    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        if ann["category_id"] == STENOSIS_CATEGORY_ID:
            anns_by_image[ann["image_id"]].append(ann)

    n_written, n_empty = 0, 0
    for img_id, img_info in images_by_id.items():
        fname = img_info["file_name"]
        stem = Path(fname).stem
        img_w, img_h = img_info["width"], img_info["height"]

        lines = []
        for ann in anns_by_image.get(img_id, []):
            xc, yc, w, h = coco_bbox_to_yolo(ann["bbox"], img_w, img_h)
            lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        label_path = label_out_dir / f"{stem}.txt"
        with open(label_path, "w") as f:
            f.write("\n".join(lines))

        if lines:
            n_written += 1
        else:
            n_empty += 1

    print(f"[{split}] wrote {n_written} label files with boxes, {n_empty} empty (background) label files.")


## 8. Build the Final YOLO Dataset Folder Structure

Ultralytics expects:

```
yolo_dataset/
    images/train/*.png  images/val/*.png  images/test/*.png
    labels/train/*.txt  labels/val/*.txt  labels/test/*.txt
```


In [ ]:
# ============================================================
# 8a. Assemble YOLO dataset directory
# ============================================================
for split in SPLITS:
    img_dst = YOLO_ROOT / "images" / split
    lbl_dst = YOLO_ROOT / "labels" / split
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    src_img_dir = PREPROC_ROOT / split
    for fname in tqdm(os.listdir(src_img_dir), desc=f"Copying images ({split})"):
        shutil.copy(src_img_dir / fname, img_dst / fname)

    convert_split_to_yolo_labels(split, coco_data[split], lbl_dst)

print("YOLO dataset assembled at:", YOLO_ROOT)


In [ ]:
# ============================================================
# 8b. Sanity check: draw YOLO-format labels back onto an image
# ============================================================
def draw_yolo_labels(img_path, label_path, ax, title=""):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    if Path(label_path).exists():
        with open(label_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        for line in lines:
            cls, xc, yc, bw, bh = map(float, line.split())
            x_min = (xc - bw / 2) * w
            y_min = (yc - bh / 2) * h
            rect = patches.Rectangle((x_min, y_min), bw * w, bh * h,
                                      linewidth=2, edgecolor="red", facecolor="none")
            ax.add_patch(rect)
    ax.set_title(title)
    ax.axis("off")

train_img_dir = YOLO_ROOT / "images" / "train"
train_lbl_dir = YOLO_ROOT / "labels" / "train"

files_with_boxes = [f for f in os.listdir(train_lbl_dir)
                     if os.path.getsize(train_lbl_dir / f) > 0][:4]

fig, axes = plt.subplots(1, len(files_with_boxes), figsize=(5 * len(files_with_boxes), 5))
for ax, lbl_fname in zip(axes, files_with_boxes):
    stem = Path(lbl_fname).stem
    img_path = train_img_dir / f"{stem}.png"
    draw_yolo_labels(img_path, train_lbl_dir / lbl_fname, ax, title=stem)
plt.tight_layout()
plt.show()


## 9. `data.yaml`

In [ ]:
# ============================================================
# 9. Write data.yaml
# ============================================================
data_yaml_content = f'''path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: stenosis
'''

data_yaml_path = WORK_ROOT / "data.yaml"
with open(data_yaml_path, "w") as f:
    f.write(data_yaml_content)

print(data_yaml_content)
print("Saved to:", data_yaml_path)


## 10. Augmentation

Ultralytics applies augmentation by default during training, which matters here because ARCADE stenosis is a small
dataset (~1500 images) and augmentation is one of the main defenses against overfitting on a dataset this size.
Choices below are tuned for *this* data rather than left at generic-photo defaults:

- **`hsv_s=0, hsv_v=0`** (changed from the original's `0.2`/`0.3`): once preprocessing is on, channels 1 and 2 are
  CLAHE and vesselness maps, not real color/brightness information. Jittering "saturation" or "value" on those
  channels doesn't correspond to any real acquisition variation — it just adds label noise. When
  `ENABLE_PREPROCESSING=False` you could reasonably turn mild `hsv_v` jitter back on to mimic exposure variation,
  but leaving both at 0 is the safer default in either case for medical images.
- **`degrees=10, translate=0.10, scale=0.30, shear=2.0`**: small geometric jitter — stenosis lesions appear at
  different positions/scales along the vessel tree, so this helps generalization without distorting anatomy into
  something implausible.
- **`perspective=0.0`**: a perspective warp has no physical analogue in a fixed-geometry X-ray projection.
- **`flipud=0.0, fliplr=0.5`**: vertical flip is anatomically implausible for a standard angiographic view; horizontal
  flip mirrors left/right coronary views and is safe.
- **`mosaic=0.8, mixup=0.1, close_mosaic=10`**: mosaic/mixup help against overfitting on a small dataset but distort
  the fine local vessel context stenosis detection relies on, so mosaic is slightly reduced from the 1.0 default and
  disabled for the final 10 epochs (`close_mosaic`) so the model fine-tunes on clean, undistorted images.


In [ ]:
# ============================================================
# 10. Augmentation configuration (passed directly to model.train())
# ============================================================
augmentation_params = dict(
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=10.0,
    translate=0.10,
    scale=0.30,
    shear=2.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.1,
    close_mosaic=10,
)

for k, v in augmentation_params.items():
    print(f"{k:12s}: {v}")


## 11. Train YOLOv8

Ultralytics automatically computes/logs box loss, classification loss, DFL loss each epoch; tracks
precision/recall/mAP@0.5/mAP@0.5:0.95 on the validation split every epoch; saves `best.pt`/`last.pt`; and writes
`results.csv`/`results.png`, confusion matrix, PR/F1 curves, and sample predictions to the run directory.

Colab free-tier sessions can disconnect before 150 epochs finish. `patience=25` (early stopping) means training
will usually stop well before that if it's not improving, but if you do get disconnected, re-run this cell with
`resume=True` on the same `project`/`name` and Ultralytics will pick up from `last.pt`.


In [ ]:
# ============================================================
# 11. Train the model
# ============================================================
RUN_NAME = f"stenosis_yolov8s_{RUN_TAG}"

model = YOLO(MODEL_WEIGHTS)

train_results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(WORK_ROOT / "runs"),
    name=RUN_NAME,
    patience=25,
    optimizer="auto",
    lr0=0.01,
    cos_lr=True,
    seed=SEED,
    pretrained=True,
    verbose=True,
    plots=True,
    **augmentation_params,
)

RUN_DIR = Path(train_results.save_dir)
print("Training run saved at:", RUN_DIR)


## 12. Training Curves (Losses + Metrics per Epoch)

In [ ]:
# ============================================================
# 12a. Show Ultralytics' auto-generated results.png
# ============================================================
results_png = RUN_DIR / "results.png"
if results_png.exists():
    img = plt.imread(results_png)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Training/Validation Losses & Metrics per Epoch")
    plt.show()
else:
    print("results.png not found at", results_png)


In [ ]:
# ============================================================
# 12b. Plot losses and metrics ourselves from results.csv
# ============================================================
results_csv = RUN_DIR / "results.csv"
hist_df = pd.read_csv(results_csv)
hist_df.columns = [c.strip() for c in hist_df.columns]
print(hist_df.tail())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

loss_cols_train = [c for c in hist_df.columns if "train/" in c and "loss" in c]
loss_cols_val = [c for c in hist_df.columns if "val/" in c and "loss" in c]
for c in loss_cols_train:
    axes[0, 0].plot(hist_df["epoch"], hist_df[c], label=c)
axes[0, 0].set_title("Training Losses"); axes[0, 0].set_xlabel("epoch"); axes[0, 0].legend()

for c in loss_cols_val:
    axes[0, 1].plot(hist_df["epoch"], hist_df[c], label=c)
axes[0, 1].set_title("Validation Losses"); axes[0, 1].set_xlabel("epoch"); axes[0, 1].legend()

pr_cols = [c for c in hist_df.columns if "precision" in c.lower() or "recall" in c.lower()]
for c in pr_cols:
    axes[1, 0].plot(hist_df["epoch"], hist_df[c], label=c)
axes[1, 0].set_title("Precision (PPV) & Recall (Sensitivity)"); axes[1, 0].set_xlabel("epoch"); axes[1, 0].legend()

map_cols = [c for c in hist_df.columns if "map" in c.lower()]
for c in map_cols:
    axes[1, 1].plot(hist_df["epoch"], hist_df[c], label=c)
axes[1, 1].set_title("mAP@0.5 & mAP@0.5:0.95"); axes[1, 1].set_xlabel("epoch"); axes[1, 1].legend()

plt.tight_layout()
plt.show()


## 13. Full Evaluation on Held-out Test Set

Loads `best.pt` and runs `model.val()` on the **test** split (never used for training or for early-stopping
decisions) to get the standard suite of detection metrics. Note for the thesis writeup: **Recall = Sensitivity** and
**Precision = PPV** — same numbers, the medical-literature names your objectives document uses.


In [ ]:
# ============================================================
# 13a. Load best checkpoint & run validation on the TEST split
# ============================================================
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
best_model = YOLO(str(BEST_WEIGHTS))

test_metrics = best_model.val(
    data=str(data_yaml_path),
    split="test",
    imgsz=IMG_SIZE,
    device=DEVICE,
    project=str(WORK_ROOT / "runs"),
    name=f"{RUN_NAME}_test_eval",
    plots=True,
)

precision = test_metrics.box.mp     # = PPV
recall = test_metrics.box.mr        # = Sensitivity
f1 = 2 * precision * recall / (precision + recall + 1e-16)

print("Test set metrics:")
print(f"  Precision / PPV:          {precision:.4f}")
print(f"  Recall / Sensitivity:     {recall:.4f}")
print(f"  F1 score:                 {f1:.4f}")
print(f"  mAP@0.5:                  {test_metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95:             {test_metrics.box.map:.4f}")

TEST_EVAL_DIR = Path(test_metrics.save_dir)
print("\nTest evaluation artifacts saved at:", TEST_EVAL_DIR)


In [ ]:
# ============================================================
# 13b. Display confusion matrix, PR curve, F1 curve (auto-generated)
# ============================================================
plot_files = {
    "Confusion Matrix": "confusion_matrix.png",
    "Confusion Matrix (Normalized)": "confusion_matrix_normalized.png",
    "Precision-Recall Curve": "PR_curve.png",
    "F1-Confidence Curve": "F1_curve.png",
    "Precision-Confidence Curve": "P_curve.png",
    "Recall-Confidence Curve": "R_curve.png",
}

available = {k: TEST_EVAL_DIR / v for k, v in plot_files.items() if (TEST_EVAL_DIR / v).exists()}

n = len(available)
fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(14, 6 * ((n + 1) // 2)))
axes = axes.flatten()
for ax, (title, path) in zip(axes, available.items()):
    ax.imshow(plt.imread(path))
    ax.set_title(title)
    ax.axis("off")
for ax in axes[len(available):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 14. Per-Class / Overall Numerical Results Table

In [ ]:
# ============================================================
# 14. Build a clean results summary table
# ============================================================
results_summary = pd.DataFrame([{
    "Run": RUN_NAME,
    "Preprocessing": ENABLE_PREPROCESSING,
    "ImgSize": IMG_SIZE,
    "Class": "stenosis",
    "Precision_PPV": round(float(precision), 4),
    "Recall_Sensitivity": round(float(recall), 4),
    "F1-score": round(float(f1), 4),
    "mAP@0.5": round(float(test_metrics.box.map50), 4),
    "mAP@0.5:0.95": round(float(test_metrics.box.map), 4),
}])

print(results_summary.to_string(index=False))

results_csv_path = WORK_ROOT / f"final_test_metrics_{RUN_TAG}.csv"
results_summary.to_csv(results_csv_path, index=False)
print("\nSaved to:", results_csv_path)
print("Run this whole notebook a second time with ENABLE_PREPROCESSING=False in Section 3, "
      "then compare the two CSVs -- that comparison is your Objective-1 ablation evidence.")


## 15. Inference on Test Images — Visualize Predictions vs Ground Truth

In [ ]:
# ============================================================
# 15a. Helper to plot GT vs Predictions for one image
# ============================================================
def plot_gt_vs_pred(image_path, label_path, model, conf_thres=0.25, ax_gt=None, ax_pred=None):
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    ax_gt.imshow(img)
    if Path(label_path).exists() and os.path.getsize(label_path) > 0:
        with open(label_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        for line in lines:
            cls, xc, yc, bw, bh = map(float, line.split())
            x_min = (xc - bw / 2) * w
            y_min = (yc - bh / 2) * h
            rect = patches.Rectangle((x_min, y_min), bw * w, bh * h,
                                      linewidth=2, edgecolor="red", facecolor="none")
            ax_gt.add_patch(rect)
    ax_gt.set_title("Ground Truth")
    ax_gt.axis("off")

    result = model.predict(source=str(image_path), imgsz=IMG_SIZE, conf=conf_thres, verbose=False)[0]
    ax_pred.imshow(img)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  linewidth=2, edgecolor="lime", facecolor="none")
        ax_pred.add_patch(rect)
        ax_pred.text(x1, max(y1 - 5, 0), f"{conf:.2f}", color="lime", fontsize=10, weight="bold")
    ax_pred.set_title("Prediction")
    ax_pred.axis("off")


In [ ]:
# ============================================================
# 15b. Run on several random test images
# ============================================================
test_img_dir = YOLO_ROOT / "images" / "test"
test_lbl_dir = YOLO_ROOT / "labels" / "test"

all_test_files = sorted(os.listdir(test_img_dir))
sample_test_files = random.sample(all_test_files, min(8, len(all_test_files)))

fig, axes = plt.subplots(len(sample_test_files), 2, figsize=(10, 5 * len(sample_test_files)))
if len(sample_test_files) == 1:
    axes = axes[None, :]

for i, fname in enumerate(sample_test_files):
    stem = Path(fname).stem
    img_path = test_img_dir / fname
    lbl_path = test_lbl_dir / f"{stem}.txt"
    plot_gt_vs_pred(img_path, lbl_path, best_model, conf_thres=0.25,
                     ax_gt=axes[i, 0], ax_pred=axes[i, 1])

plt.tight_layout()
plt.show()


## 16. Manual IoU-based Numerical Analysis (Cross-check)

Independent of Ultralytics' internal metric code: for each test image, greedily match predicted boxes to
ground-truth boxes at IoU ≥ 0.5 and tally TP/FP/FN, then derive Precision/Recall/F1 by hand. If this disagrees
substantially with Section 13's numbers, that's worth investigating (different NMS/confidence handling) rather than
reporting whichever number is higher — a thesis defense will ask which is which and why they differ.


In [ ]:
# ============================================================
# 16. Manual IoU-based TP/FP/FN tally
# ============================================================
def yolo_line_to_xyxy(line, img_w, img_h):
    cls, xc, yc, bw, bh = map(float, line.split())
    x1 = (xc - bw / 2) * img_w
    y1 = (yc - bh / 2) * img_h
    x2 = (xc + bw / 2) * img_w
    y2 = (yc + bh / 2) * img_h
    return [x1, y1, x2, y2]

def iou_xyxy(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    inter_x1, inter_y1 = max(xa1, xb1), max(ya1, yb1)
    inter_x2, inter_y2 = min(xa2, xb2), min(ya2, yb2)
    inter_w, inter_h = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0

def evaluate_iou_matching(model, img_dir, lbl_dir, conf_thres=0.25, iou_thres=0.5):
    TP, FP, FN = 0, 0, 0
    for fname in tqdm(sorted(os.listdir(img_dir)), desc="Manual IoU eval"):
        stem = Path(fname).stem
        img_path = img_dir / fname
        lbl_path = lbl_dir / f"{stem}.txt"

        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]

        gt_boxes = []
        if lbl_path.exists() and os.path.getsize(lbl_path) > 0:
            with open(lbl_path) as f:
                gt_boxes = [yolo_line_to_xyxy(l.strip(), w, h) for l in f if l.strip()]

        result = model.predict(source=str(img_path), imgsz=IMG_SIZE, conf=conf_thres, verbose=False)[0]
        pred_boxes = [box.xyxy[0].cpu().numpy().tolist() for box in result.boxes]

        matched_gt = set()
        for pb in pred_boxes:
            best_iou, best_idx = 0.0, -1
            for idx, gb in enumerate(gt_boxes):
                if idx in matched_gt:
                    continue
                cur_iou = iou_xyxy(pb, gb)
                if cur_iou > best_iou:
                    best_iou, best_idx = cur_iou, idx
            if best_iou >= iou_thres:
                TP += 1
                matched_gt.add(best_idx)
            else:
                FP += 1
        FN += len(gt_boxes) - len(matched_gt)

    precision_m = TP / (TP + FP + 1e-16)
    recall_m = TP / (TP + FN + 1e-16)
    f1_m = 2 * precision_m * recall_m / (precision_m + recall_m + 1e-16)
    return {"TP": TP, "FP": FP, "FN": FN, "Precision": precision_m, "Recall": recall_m, "F1": f1_m}

manual_metrics = evaluate_iou_matching(best_model, test_img_dir, test_lbl_dir, conf_thres=0.25, iou_thres=0.5)
print("Manual IoU-based matching results (conf>=0.25, IoU>=0.5):")
for k, v in manual_metrics.items():
    print(f"  {k}: {v}")


## 17. Final Combined Results Summary

In [ ]:
# ============================================================
# 17. Final combined numerical results (Ultralytics vs manual cross-check)
# ============================================================
final_summary = pd.DataFrame([
    {
        "Method": "Ultralytics model.val() (test split)",
        "Precision_PPV": round(float(precision), 4),
        "Recall_Sensitivity": round(float(recall), 4),
        "F1-score": round(float(f1), 4),
        "mAP@0.5": round(float(test_metrics.box.map50), 4),
        "mAP@0.5:0.95": round(float(test_metrics.box.map), 4),
    },
    {
        "Method": "Manual IoU matching (conf>=0.25, IoU>=0.5)",
        "Precision_PPV": round(manual_metrics["Precision"], 4),
        "Recall_Sensitivity": round(manual_metrics["Recall"], 4),
        "F1-score": round(manual_metrics["F1"], 4),
        "mAP@0.5": "-",
        "mAP@0.5:0.95": "-",
    },
])

print(final_summary.to_string(index=False))
final_summary_path = WORK_ROOT / f"final_combined_metrics_{RUN_TAG}.csv"
final_summary.to_csv(final_summary_path, index=False)
print("\nSaved to:", final_summary_path)


## 18. Save Final Model (to Google Drive)

In [ ]:
# ============================================================
# 18. Copy weights + metrics to Google Drive so they survive the session
# ============================================================
run_output_dir = DRIVE_OUTPUT_DIR / RUN_NAME
run_output_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(BEST_WEIGHTS, run_output_dir / "best.pt")
shutil.copy(results_csv_path, run_output_dir / results_csv_path.name)
shutil.copy(final_summary_path, run_output_dir / final_summary_path.name)
if results_png.exists():
    shutil.copy(results_png, run_output_dir / "results.png")

print("Saved to Google Drive at:", run_output_dir)
print("(If you skipped the Drive-mount cell in Section 2, download the files instead "
      "from the Colab file browser under", WORK_ROOT, ")")


## Notes & Next Steps

- **Ablation to run for the thesis**: set `ENABLE_PREPROCESSING = False` in Section 3 and re-run the whole notebook
  end-to-end as a second run. Comparing `final_test_metrics_raw.csv` vs `final_test_metrics_enhanced.csv` is your
  direct evidence that the preprocessing step improves accuracy — report both numbers, not just the better one.
- If recall/sensitivity is still low on small or faint lesions after this pass, the next accuracy levers to try, in
  order of effort: (1) `yolov8m.pt` instead of `yolov8s.pt` (more capacity, watch val loss for overfitting since the
  dataset is small), (2) `imgsz=1280` (more GPU memory needed, drop `BATCH_SIZE` accordingly), (3) k-fold
  cross-validation instead of a single train/val/test split, since a single split can be noisy for a dataset this
  size — useful for reporting a mean ± std rather than one number, though the official ARCADE split is what's
  comparable to the literature table in the proposal.
- Objective 1 (this notebook) should be locked down — a real, reported accuracy number with an honest ablation —
  before moving to Objective 2 (Grad-CAM / attention-map explainability module) and Objective 3 (cross-dataset
  evaluation on DCA1 or CADICA). Both build directly on `best.pt` from this notebook, so nothing here needs to be
  redone for those stages.
